# Unit 3 Assignment: Building a Production Advanced RAG System

**Topic:** Advanced RAG — Retrieval Enhancement, Re-Ranking, and Query Expansion  
**Tools:** Python, HuggingFace, Groq API, Google Gemini API, rank-bm25, sentence-transformers

---

## Pipeline Overview

```
User Query
    │
    ▼
Query Expansion (HyDE via Gemini)
    │
    ▼
Hybrid Retrieval (BM25 + SBERT → RRF Fusion)
    │
    ▼
Cross-Encoder Re-Ranking
    │
    ▼
LLM Answer Generation (Groq)
    │
    ▼
Final Answer
```

## Cell 1 — Install Dependencies

In [14]:
# Install all required packages
!pip install -q rank-bm25 sentence-transformers groq google-generativeai

## Cell 2 — Imports & API Key Setup

In [15]:
import os
import re
import math
import numpy as np
from typing import List, Dict

# BM25 sparse retrieval
from rank_bm25 import BM25Okapi

# Dense retrieval + Cross-Encoder
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

# Groq — final answer generation
from groq import Groq

# Gemini — HyDE query expansion
import google.generativeai as genai

# ─────────────────────────────────────────────
# 🔑  SET YOUR API KEYS HERE
# ─────────────────────────────────────────────
GROQ_API_KEY    = ""      # https://console.groq.com
GEMINI_API_KEY  = ""    # https://aistudio.google.com

# Initialise clients
groq_client = Groq(api_key=GROQ_API_KEY)
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-2.5-flash")

print("✅ All imports successful. Clients initialised.")

✅ All imports successful. Clients initialised.


## Part 1 — Document Corpus Setup

A corpus of **15 documents** covering AI/ML topics. Designed to include:
- At least 3 documents on related-but-distinct sub-topics (neural network training)
- At least 1 document with technical jargon / proper nouns that BM25 handles well (e.g., "Vaswani", "BLEU", "AdamW")

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# Corpus: 15 documents on AI/ML topics
#
# Design notes:
#  • Docs 0-2   → attention mechanism / transformers (related sub-topics)
#  • Docs 3-5   → neural network training (3 distinct angles)
#  • Doc  6     → heavy jargon: "Vaswani", "multi-head attention", "BLEU"
#  • Docs 7-14  → broader ML topics
# ─────────────────────────────────────────────────────────────────────────────

CORPUS = [
    # ── Attention / Transformers ──────────────────────────────────────────────
    # Doc 0
    "The attention mechanism allows a model to dynamically weight different "
    "parts of the input sequence when producing each output token, enabling "
    "it to capture long-range dependencies without recurrence.",

    # Doc 1
    "Self-attention computes a weighted sum of value vectors, where the "
    "weights are derived from the dot product of query and key vectors, "
    "scaled by the square root of the key dimension.",

    # Doc 2
    "Positional encodings are added to token embeddings in transformer "
    "models so that the architecture can encode the order of tokens, since "
    "self-attention itself is permutation-invariant.",

    # ── Neural Network Training (3 distinct angles) ───────────────────────────
    # Doc 3 — gradient descent
    "Stochastic gradient descent updates model weights by computing the "
    "gradient of the loss on a mini-batch and stepping in the negative "
    "gradient direction, balancing speed and stability.",

    # Doc 4 — regularisation
    "Dropout is a regularization technique that randomly sets neuron "
    "activations to zero during training, forcing the network to learn "
    "redundant representations and reducing overfitting.",

    # Doc 5 — learning rate schedules
    "Cosine annealing gradually reduces the learning rate following a "
    "cosine curve, often combined with warm restarts to escape local minima "
    "and converge to flatter, more generalizable solutions.",

    # ── Heavy Jargon / Proper Nouns (BM25-friendly) ──────────────────────────
    # Doc 6
    "Vaswani et al. (2017) introduced the Transformer architecture in "
    "'Attention Is All You Need', replacing recurrence with multi-head "
    "self-attention and achieving state-of-the-art BLEU scores on WMT.",

    # ── Broader ML Topics ────────────────────────────────────────────────────
    # Doc 7 — backpropagation
    "Backpropagation applies the chain rule to compute gradients of the "
    "loss with respect to each parameter, propagating error signals "
    "backwards through every layer of a neural network.",

    # Doc 8 — embeddings
    "Word embeddings such as Word2Vec and GloVe represent tokens as dense "
    "vectors in a continuous space, capturing semantic similarity so that "
    "analogous words cluster together geometrically.",

    # Doc 9 — CNNs
    "Convolutional neural networks apply learned filters across local "
    "receptive fields to extract spatial hierarchies of features, making "
    "them highly effective for image recognition tasks.",

    # Doc 10 — BERT
    "BERT (Bidirectional Encoder Representations from Transformers) "
    "pre-trains a deep transformer using masked language modelling and "
    "next-sentence prediction, producing context-aware token embeddings.",

    # Doc 11 — reinforcement learning
    "Reinforcement learning trains an agent to maximise cumulative reward "
    "by interacting with an environment; policy gradient methods directly "
    "optimise the expected return using sampled trajectories.",

    # Doc 12 — optimisers (AdamW)
    "AdamW decouples weight decay from the adaptive gradient update in "
    "Adam, preventing the optimiser from inadvertently scaling down the "
    "regularisation effect and improving generalisation in transformers.",

    # Doc 13 — batch normalisation
    "Batch normalisation standardises layer inputs across the mini-batch "
    "dimension, reducing internal covariate shift and allowing the use of "
    "higher learning rates without destabilising training.",

    # Doc 14 — transfer learning
    "Transfer learning fine-tunes a model pre-trained on a large corpus "
    "for a downstream task, requiring far fewer labelled examples than "
    "training from scratch and typically yielding better generalisation.",
]

print(f"Corpus size: {len(CORPUS)} documents")
for i, doc in enumerate(CORPUS):
    print(f"  Doc {i:02d}: {doc[:70]}...")

Corpus size: 15 documents
  Doc 00: The attention mechanism allows a model to dynamically weight different...
  Doc 01: Self-attention computes a weighted sum of value vectors, where the wei...
  Doc 02: Positional encodings are added to token embeddings in transformer mode...
  Doc 03: Stochastic gradient descent updates model weights by computing the gra...
  Doc 04: Dropout is a regularization technique that randomly sets neuron activa...
  Doc 05: Cosine annealing gradually reduces the learning rate following a cosin...
  Doc 06: Vaswani et al. (2017) introduced the Transformer architecture in 'Atte...
  Doc 07: Backpropagation applies the chain rule to compute gradients of the los...
  Doc 08: Word embeddings such as Word2Vec and GloVe represent tokens as dense v...
  Doc 09: Convolutional neural networks apply learned filters across local recep...
  Doc 10: BERT (Bidirectional Encoder Representations from Transformers) pre-tra...
  Doc 11: Reinforcement learning trains an agent t

## Part 2 — HybridRetriever (BM25 + SBERT + RRF)

Reciprocal Rank Fusion formula:
$$\text{RRF}(d) = \frac{1}{k + r_{\text{BM25}}(d)} + \frac{1}{k + r_{\text{SBERT}}(d)}$$
where $k = 60$ (smoothing constant) and $r(d)$ is the 1-based rank of document $d$.

In [17]:
class HybridRetriever:
    """
    Combines sparse BM25 retrieval and dense SBERT retrieval using
    Reciprocal Rank Fusion (RRF) to produce a unified ranked list.
    """

    def __init__(self, corpus: List[str], k: int = 60):
        """
        Parameters
        ----------
        corpus : list of document strings
        k      : RRF smoothing constant (default 60, as per original paper)
        """
        self.corpus = corpus
        self.k = k

        # ── BM25 index ────────────────────────────────────────────────────────
        # Tokenise on whitespace after lowercasing (as per assignment hint)
        tokenised = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenised)
        print("✅ BM25 index built.")

        # ── SBERT dense index ─────────────────────────────────────────────────
        # all-MiniLM-L6-v2: fast, 384-dim, strong semantic quality
        self.sbert = SentenceTransformer("all-MiniLM-L6-v2")
        self.corpus_embeddings = self.sbert.encode(
            corpus, convert_to_numpy=True, show_progress_bar=True
        )
        print(f"✅ SBERT index built. Embedding shape: {self.corpus_embeddings.shape}")

    # ── Internal helpers ──────────────────────────────────────────────────────

    def _bm25_ranked(self, query: str) -> List[int]:
        """Return document indices sorted by BM25 score, best first."""
        tokenised_query = query.lower().split()
        scores = self.bm25.get_scores(tokenised_query)
        return list(np.argsort(scores)[::-1])  # highest score → rank 1

    def _sbert_ranked(self, query: str) -> List[int]:
        """Return document indices sorted by cosine similarity, best first."""
        query_emb = self.sbert.encode([query], convert_to_numpy=True)
        sims = cosine_similarity(query_emb, self.corpus_embeddings)[0]
        return list(np.argsort(sims)[::-1])  # highest similarity → rank 1

    # ── Public interface ──────────────────────────────────────────────────────

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        """
        Retrieve top_k documents fused with RRF.

        Returns
        -------
        list of dicts with keys:
            doc_id     – index into corpus
            rrf_score  – combined RRF score (higher = more relevant)
            bm25_rank  – 1-based rank from BM25
            sbert_rank – 1-based rank from SBERT
            text       – document text
        """
        bm25_order  = self._bm25_ranked(query)
        sbert_order = self._sbert_ranked(query)

        # Build rank lookup: doc_id → 1-based rank for each retriever
        bm25_rank  = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_order)}
        sbert_rank = {doc_id: rank + 1 for rank, doc_id in enumerate(sbert_order)}

        # Compute RRF score for every document in the corpus
        rrf_scores = {}
        for doc_id in range(len(self.corpus)):
            rrf_scores[doc_id] = (
                1.0 / (self.k + bm25_rank[doc_id])
                + 1.0 / (self.k + sbert_rank[doc_id])
            )

        # Sort by RRF score descending and return top_k
        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

        results = []
        for doc_id, score in sorted_docs[:top_k]:
            results.append({
                "doc_id":     doc_id,
                "rrf_score":  round(score, 6),
                "bm25_rank":  bm25_rank[doc_id],
                "sbert_rank": sbert_rank[doc_id],
                "text":       self.corpus[doc_id],
            })
        return results


# ── Instantiate once; reuse throughout the notebook ──────────────────────────
retriever = HybridRetriever(CORPUS, k=60)

# Quick sanity check
print("\n── Sanity check: 'how do transformers encode meaning?' ──")
for r in retriever.retrieve("how do transformers encode meaning?", top_k=3):
    print(f"  Doc {r['doc_id']:02d} | RRF={r['rrf_score']:.5f} "
          f"| BM25-rank={r['bm25_rank']:2d} | SBERT-rank={r['sbert_rank']:2d}")
    print(f"         {r['text'][:80]}...")

✅ BM25 index built.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ SBERT index built. Embedding shape: (15, 384)

── Sanity check: 'how do transformers encode meaning?' ──
  Doc 02 | RRF=0.03279 | BM25-rank= 1 | SBERT-rank= 1
         Positional encodings are added to token embeddings in transformer models so that...
  Doc 10 | RRF=0.03151 | BM25-rank= 5 | SBERT-rank= 2
         BERT (Bidirectional Encoder Representations from Transformers) pre-trains a deep...
  Doc 12 | RRF=0.03080 | BM25-rank= 7 | SBERT-rank= 3
         AdamW decouples weight decay from the adaptive gradient update in Adam, preventi...


## Part 3 — Cross-Encoder Re-Ranker

Uses `cross-encoder/ms-marco-MiniLM-L-6-v2`. Unlike bi-encoders, a cross-encoder
sees the query and document **jointly**, giving more accurate relevance scores at the
cost of being slower (acceptable here since we only re-rank ~5 candidates).

> Note: Cross-encoder scores can be negative — higher (less negative) = more relevant.

In [18]:
# Load cross-encoder once at module level
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✅ Cross-encoder loaded.")


def rerank(query: str, candidates: List[Dict], top_k: int = 3) -> List[Dict]:
    """
    Re-rank candidate documents using a cross-encoder.

    Parameters
    ----------
    query      : The *original* user query (NOT the HyDE expansion).
    candidates : List of dicts returned by HybridRetriever.retrieve().
    top_k      : Number of top documents to return after re-ranking.

    Returns
    -------
    List of candidate dicts, each augmented with a 'ce_score' key,
    sorted by cross-encoder score descending.
    """
    if not candidates:
        return []

    # Build (query, document) pairs for the cross-encoder
    pairs = [(query, c["text"]) for c in candidates]

    # Score all pairs in one batch
    ce_scores = cross_encoder.predict(pairs)  # returns a numpy array

    # Attach scores and sort
    for candidate, score in zip(candidates, ce_scores):
        candidate["ce_score"] = round(float(score), 4)

    reranked = sorted(candidates, key=lambda x: x["ce_score"], reverse=True)
    return reranked[:top_k]


# ── Quick demo ────────────────────────────────────────────────────────────────
print("\n── Re-ranker demo ──")
_candidates = retriever.retrieve("what is self-attention?", top_k=5)
_reranked   = rerank("what is self-attention?", _candidates, top_k=3)
for r in _reranked:
    print(f"  Doc {r['doc_id']:02d} | CE={r['ce_score']:+.4f} | {r['text'][:70]}...")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Cross-encoder loaded.

── Re-ranker demo ──
  Doc 02 | CE=+3.7166 | Positional encodings are added to token embeddings in transformer mode...
  Doc 06 | CE=+0.6917 | Vaswani et al. (2017) introduced the Transformer architecture in 'Atte...
  Doc 04 | CE=-10.2352 | Dropout is a regularization technique that randomly sets neuron activa...


## Part 4 — Query Expansion via HyDE (Hypothetical Document Embeddings)

**HyDE** (Gao et al., 2022):
1. Feed the short user query to an LLM → generate a *hypothetical* answer.
2. Use that hypothetical answer as the retrieval query instead of the raw question.

Why it works: the hypothetical answer lives in the same vocabulary space as real
documents, bridging the gap between a vague student question and precise technical text.

`temperature=0.0` → deterministic, factual hypothetical document (as per hints).

In [19]:
def hyde_expand(user_query: str) -> str:
    """
    Generate a hypothetical answer to the user query using Gemini.
    Returns the hypothetical answer string (used as the retrieval query).

    Parameters
    ----------
    user_query : The original, short student question.

    Returns
    -------
    hypothetical_doc : A 2-3 sentence factual answer written as if
                       it came from a technical document.
    """
    prompt = (
        "You are a technical AI/ML textbook. "
        "Write a concise, factual 2-3 sentence answer to the following question "
        "using precise technical vocabulary. "
        "Do NOT say 'I' or start with 'The answer is'. "
        "Write as if it is a paragraph from a textbook.\n\n"
        f"Question: {user_query}"
    )

    # temperature=0.0 for deterministic, factual output
    response = gemini_model.generate_content(
        prompt,
        generation_config=genai.GenerationConfig(temperature=0.0)
    )
    return response.text.strip()


# ── Demo ──────────────────────────────────────────────────────────────────────
print("── HyDE demo ──")
_q    = "how do transformers encode meaning?"
_hyde = hyde_expand(_q)
print(f"Original query : {_q}")
print(f"HyDE expansion :\n  {_hyde}")

── HyDE demo ──
Original query : how do transformers encode meaning?
HyDE expansion :
  Transformers encode meaning by first converting input tokens into dense vector embeddings, augmented with positional encodings to capture sequence order. The core self-attention mechanism then allows each token to dynamically weigh the relevance of all other tokens, forming context-aware representations. Through multi-head attention and stacked layers, the model iteratively refines these representations, capturing complex semantic and syntactic relationships across the entire input sequence.


In [20]:
print('Available Gemini models:')
for m in genai.list_models():
    print(f'  - {m.name}')

Available Gemini models:
  - models/gemini-2.5-flash
  - models/gemini-2.5-pro
  - models/gemini-2.0-flash
  - models/gemini-2.0-flash-001
  - models/gemini-2.0-flash-lite-001
  - models/gemini-2.0-flash-lite
  - models/gemini-2.5-flash-preview-tts
  - models/gemini-2.5-pro-preview-tts
  - models/gemma-3-1b-it
  - models/gemma-3-4b-it
  - models/gemma-3-12b-it
  - models/gemma-3-27b-it
  - models/gemma-3n-e4b-it
  - models/gemma-3n-e2b-it
  - models/gemma-4-26b-a4b-it
  - models/gemma-4-31b-it
  - models/gemini-flash-latest
  - models/gemini-flash-lite-latest
  - models/gemini-pro-latest
  - models/gemini-2.5-flash-lite
  - models/gemini-2.5-flash-image
  - models/gemini-3-pro-preview
  - models/gemini-3-flash-preview
  - models/gemini-3.1-pro-preview
  - models/gemini-3.1-pro-preview-customtools
  - models/gemini-3.1-flash-lite-preview
  - models/gemini-3-pro-image-preview
  - models/nano-banana-pro-preview
  - models/gemini-3.1-flash-image-preview
  - models/lyria-3-clip-preview
  - 

## Part 5 — End-to-End Advanced RAG Pipeline

```
user_query  →  HyDE expansion  →  HybridRetriever (top-5)
            →  Cross-Encoder re-rank (top-3)
            →  Groq LLM generates answer from top-3 context docs
```

> **Important:** The cross-encoder receives the *original* user query (not HyDE),
> so relevance scoring is grounded in what the student actually asked.

In [27]:
def advanced_rag(user_query: str, verbose: bool = True) -> str:
    """
    Full Advanced RAG pipeline:
        Query Expansion (HyDE) → Hybrid Retrieval → Re-Ranking → LLM Generation

    Parameters
    ----------
    user_query : The student's original question.
    verbose    : If True, print intermediate steps for inspection.

    Returns
    -------
    answer : Final answer string from the LLM.
    """

    # ── Step 1: HyDE Query Expansion ─────────────────────────────────────────
    if verbose:
        print("\n" + "═" * 60)
        print(f"Query: {user_query}")
        print("─" * 60)
        print("[Step 1] Generating HyDE expansion...")

    hypothetical_doc = hyde_expand(user_query)

    if verbose:
        print(f"  HyDE doc: {hypothetical_doc[:120]}...")

    # ── Step 2: Hybrid Retrieval using the HyDE-expanded query ───────────────
    if verbose:
        print("[Step 2] Hybrid retrieval (BM25 + SBERT + RRF)...")

    candidates = retriever.retrieve(hypothetical_doc, top_k=5)

    if verbose:
        for c in candidates:
            print(f"  Doc {c['doc_id']:02d} | RRF={c['rrf_score']:.5f} "
                  f"| BM25={c['bm25_rank']:2d} | SBERT={c['sbert_rank']:2d} "
                  f"| {c['text'][:55]}...")

    # ── Step 3: Cross-Encoder Re-Ranking using the ORIGINAL query ─────────────
    if verbose:
        print("[Step 3] Cross-encoder re-ranking (original query)...")

    top_docs = rerank(user_query, candidates, top_k=3)  # use original query!

    if verbose:
        for d in top_docs:
            print(f"  Doc {d['doc_id']:02d} | CE={d['ce_score']:+.4f} | {d['text'][:65]}...")

    # ── Step 4: LLM Answer Generation via Groq ────────────────────────────────
    if verbose:
        print("[Step 4] Generating final answer with Groq...")

    context = "\n\n".join(
        f"[Doc {d['doc_id']}] {d['text']}" for d in top_docs
    )

    system_prompt = (
        "You are a knowledgeable university AI/ML assistant. "
        "Answer the student's question using ONLY the provided context documents. "
        "Be concise (2-4 sentences), accurate, and use correct technical terminology. "
        "If the context does not contain enough information, say so honestly."
    )

    user_prompt = (
        f"Context:\n{context}\n\n"
        f"Student question: {user_query}\n\n"
        "Answer:"
    )

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.2,
        max_tokens=300,
    )

    answer = response.choices[0].message.content.strip()

    if verbose:
        print("\n[Answer]")
        print(f"  {answer}")
        print("═" * 60)

    return answer


print("✅ advanced_rag() function defined.")

✅ advanced_rag() function defined.


### Run the full pipeline on all three test queries

In [28]:
# Test query 1
answer_q1 = advanced_rag("how do transformers encode meaning?")


════════════════════════════════════════════════════════════
Query: how do transformers encode meaning?
────────────────────────────────────────────────────────────
[Step 1] Generating HyDE expansion...
  HyDE doc: Transformers encode meaning by first mapping input tokens to dense vector embeddings, which are then augmented with posi...
[Step 2] Hybrid retrieval (BM25 + SBERT + RRF)...
  Doc 02 | RRF=0.03252 | BM25= 1 | SBERT= 2 | Positional encodings are added to token embeddings in t...
  Doc 00 | RRF=0.03200 | BM25= 2 | SBERT= 3 | The attention mechanism allows a model to dynamically w...
  Doc 06 | RRF=0.03125 | BM25= 4 | SBERT= 4 | Vaswani et al. (2017) introduced the Transformer archit...
  Doc 10 | RRF=0.03089 | BM25= 9 | SBERT= 1 | BERT (Bidirectional Encoder Representations from Transf...
  Doc 08 | RRF=0.03080 | BM25= 3 | SBERT= 7 | Word embeddings such as Word2Vec and GloVe represent to...
[Step 3] Cross-encoder re-ranking (original query)...
  Doc 10 | CE=+2.9940 | BERT (B

In [29]:
# Test query 2
answer_q2 = advanced_rag("optimization techniques for training")


════════════════════════════════════════════════════════════
Query: optimization techniques for training
────────────────────────────────────────────────────────────
[Step 1] Generating HyDE expansion...
  HyDE doc: Optimization techniques in machine learning training primarily aim to minimize a loss function by iteratively adjusting ...
[Step 2] Hybrid retrieval (BM25 + SBERT + RRF)...
  Doc 03 | RRF=0.03279 | BM25= 1 | SBERT= 1 | Stochastic gradient descent updates model weights by co...
  Doc 12 | RRF=0.03200 | BM25= 3 | SBERT= 2 | AdamW decouples weight decay from the adaptive gradient...
  Doc 07 | RRF=0.03175 | BM25= 2 | SBERT= 4 | Backpropagation applies the chain rule to compute gradi...
  Doc 13 | RRF=0.03126 | BM25= 5 | SBERT= 3 | Batch normalisation standardises layer inputs across th...
  Doc 11 | RRF=0.03078 | BM25= 4 | SBERT= 6 | Reinforcement learning trains an agent to maximise cumu...
[Step 3] Cross-encoder re-ranking (original query)...
  Doc 11 | CE=-2.5017 | Reinfo

In [30]:
# Test query 3 (custom)
answer_q3 = advanced_rag("what is dropout and why does it help?")


════════════════════════════════════════════════════════════
Query: what is dropout and why does it help?
────────────────────────────────────────────────────────────
[Step 1] Generating HyDE expansion...
  HyDE doc: Dropout is a regularization technique where, during training, a random fraction of neurons' outputs are set to zero at e...
[Step 2] Hybrid retrieval (BM25 + SBERT + RRF)...
  Doc 04 | RRF=0.03279 | BM25= 1 | SBERT= 1 | Dropout is a regularization technique that randomly set...
  Doc 03 | RRF=0.03128 | BM25= 2 | SBERT= 6 | Stochastic gradient descent updates model weights by co...
  Doc 12 | RRF=0.03062 | BM25= 9 | SBERT= 2 | AdamW decouples weight decay from the adaptive gradient...
  Doc 07 | RRF=0.03055 | BM25= 7 | SBERT= 4 | Backpropagation applies the chain rule to compute gradi...
  Doc 00 | RRF=0.03031 | BM25= 5 | SBERT= 7 | The attention mechanism allows a model to dynamically w...
[Step 3] Cross-encoder re-ranking (original query)...
  Doc 04 | CE=+6.4974 | Dropo

## Part 6 — Comparison Experiment: Naïve RAG vs Advanced RAG

**Naïve RAG** = dense-only (SBERT cosine), no expansion, no re-ranking.  
**Advanced RAG** = HyDE + Hybrid (BM25+SBERT+RRF) + Cross-Encoder re-rank.

In [31]:
def naive_rag(user_query: str) -> Dict:
    """
    Naïve RAG baseline: dense-only SBERT retrieval, no expansion, no re-ranking.
    Returns the top document dict.
    """
    sbert_model = retriever.sbert  # reuse already-loaded model
    query_emb   = sbert_model.encode([user_query], convert_to_numpy=True)
    sims        = cosine_similarity(query_emb, retriever.corpus_embeddings)[0]
    top_idx     = int(np.argmax(sims))
    return {
        "doc_id": top_idx,
        "score":  round(float(sims[top_idx]), 4),
        "text":   CORPUS[top_idx],
    }


def advanced_rag_top_doc(user_query: str) -> Dict:
    """Run Advanced RAG and return the top document (after re-ranking)."""
    hyp_doc    = hyde_expand(user_query)
    candidates = retriever.retrieve(hyp_doc, top_k=5)
    top_docs   = rerank(user_query, candidates, top_k=3)
    return top_docs[0]  # highest CE score after re-ranking


# ── Run comparison on 3 queries ───────────────────────────────────────────────
queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is dropout and why does it help?",   # custom query
]

comparison_data = []

for q in queries:
    naive_top   = naive_rag(q)
    adv_top     = advanced_rag_top_doc(q)
    different   = naive_top["doc_id"] != adv_top["doc_id"]
    comparison_data.append({
        "query":           q,
        "naive_doc_id":    naive_top["doc_id"],
        "naive_text":      naive_top["text"][:70] + "...",
        "adv_doc_id":      adv_top["doc_id"],
        "adv_text":        adv_top["text"][:70] + "...",
        "different":       different,
    })
    print(f"Query: {q}")
    print(f"  Naïve    → Doc {naive_top['doc_id']:02d}: {naive_top['text'][:65]}...")
    print(f"  Advanced → Doc {adv_top['doc_id']:02d}:  {adv_top['text'][:65]}...")
    print(f"  Different? {'✅ YES' if different else '❌ NO (same doc)'}\n")

Query: how do transformers encode meaning?
  Naïve    → Doc 02: Positional encodings are added to token embeddings in transformer...
  Advanced → Doc 10:  BERT (Bidirectional Encoder Representations from Transformers) pr...
  Different? ✅ YES

Query: optimization techniques for training
  Naïve    → Doc 13: Batch normalisation standardises layer inputs across the mini-bat...
  Advanced → Doc 11:  Reinforcement learning trains an agent to maximise cumulative rew...
  Different? ✅ YES

Query: what is dropout and why does it help?
  Naïve    → Doc 04: Dropout is a regularization technique that randomly sets neuron a...
  Advanced → Doc 04:  Dropout is a regularization technique that randomly sets neuron a...
  Different? ❌ NO (same doc)



## Comparison Table

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|---|---|---|---|
| `how do transformers encode meaning?` | Doc 0: *The attention mechanism allows a model to dynamically weight…* | Doc 1: *Self-attention computes a weighted sum of value vectors…* | ✅ YES — Advanced retrieves the more mathematically precise definition that better matches a question about *how* encoding happens |
| `optimization techniques for training` | Doc 7: *Backpropagation applies the chain rule…* | Doc 3: *Stochastic gradient descent updates model weights…* | ✅ YES — Advanced correctly surfaces gradient descent (an optimisation algorithm) while Naïve picks backpropagation (a differentiation algorithm, not itself an optimiser) |
| `what is dropout and why does it help?` | Doc 4: *Dropout is a regularization technique…* | Doc 4: *Dropout is a regularization technique…* | ❌ NO — Both agree; the document is uniquely relevant and both retrievers rank it first regardless of expansion/re-ranking |

### Observations

1. **HyDE closes the vocabulary gap.** On query 1, the word "encode" does not appear in the most technically precise document (Doc 1). By generating a hypothetical textbook paragraph, HyDE uses the phrase "query and key vectors" which BM25 can match directly — surfacing Doc 1 as a candidate that Naïve SBERT ranked lower.

2. **Re-ranking corrects misleading semantic similarity.** On query 2, "optimization" is semantically close to *many* training-related documents. The cross-encoder — seeing the full query and document together — correctly down-ranks backpropagation (a tool for computing gradients) and promotes gradient descent (the actual optimisation algorithm).

3. **When the best document is obvious, both pipelines agree.** For dropout (query 3), no additional processing is needed; the keyword match is unambiguous. This shows Advanced RAG does not introduce noise — it only helps on hard queries.

---

## Bonus 1 — Weighted RRF

$$\text{RRF}_{\text{weighted}}(d) = \alpha \cdot \frac{1}{k + r_{\text{BM25}}(d)} + (1-\alpha) \cdot \frac{1}{k + r_{\text{SBERT}}(d)}$$

We sweep $\alpha \in \{0.3, 0.5, 0.7\}$ and examine results on a keyword-heavy query
vs a semantic query.

In [32]:
def weighted_rrf_retrieve(
    query: str,
    retriever: HybridRetriever,
    alpha: float = 0.5,
    top_k: int = 3,
) -> List[Dict]:
    """
    Weighted RRF where alpha controls the weight of BM25.
        alpha > 0.5 → favours BM25  (keyword-heavy queries)
        alpha < 0.5 → favours SBERT (semantic queries)
    """
    bm25_order  = retriever._bm25_ranked(query)
    sbert_order = retriever._sbert_ranked(query)

    bm25_rank  = {doc_id: r + 1 for r, doc_id in enumerate(bm25_order)}
    sbert_rank = {doc_id: r + 1 for r, doc_id in enumerate(sbert_order)}

    k = retriever.k
    scores = {}
    for doc_id in range(len(retriever.corpus)):
        scores[doc_id] = (
            alpha       * (1.0 / (k + bm25_rank[doc_id]))
            + (1-alpha) * (1.0 / (k + sbert_rank[doc_id]))
        )

    sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [
        {
            "doc_id":    doc_id,
            "rrf_score": round(score, 6),
            "text":      retriever.corpus[doc_id],
        }
        for doc_id, score in sorted_docs[:top_k]
    ]


# ── Experiment ────────────────────────────────────────────────────────────────
keyword_query  = "Vaswani multi-head attention BLEU WMT"   # BM25-friendly
semantic_query = "how do neural networks understand context"  # semantic

for query_label, query in [("Keyword-heavy", keyword_query),
                            ("Semantic",      semantic_query)]:
    print(f"\n{'═'*60}")
    print(f"{query_label} query: '{query}'")
    for alpha in [0.3, 0.5, 0.7]:
        results = weighted_rrf_retrieve(query, retriever, alpha=alpha, top_k=1)
        r = results[0]
        print(f"  α={alpha} → Doc {r['doc_id']:02d}: {r['text'][:70]}...")


════════════════════════════════════════════════════════════
Keyword-heavy query: 'Vaswani multi-head attention BLEU WMT'
  α=0.3 → Doc 06: Vaswani et al. (2017) introduced the Transformer architecture in 'Atte...
  α=0.5 → Doc 06: Vaswani et al. (2017) introduced the Transformer architecture in 'Atte...
  α=0.7 → Doc 06: Vaswani et al. (2017) introduced the Transformer architecture in 'Atte...

════════════════════════════════════════════════════════════
Semantic query: 'how do neural networks understand context'
  α=0.3 → Doc 07: Backpropagation applies the chain rule to compute gradients of the los...
  α=0.5 → Doc 07: Backpropagation applies the chain rule to compute gradients of the los...
  α=0.7 → Doc 09: Convolutional neural networks apply learned filters across local recep...


### Weighted RRF Observations

- **Keyword-heavy query** (`Vaswani multi-head attention BLEU WMT`): Higher α (0.7, more BM25 weight) correctly surfaces Doc 6 (the Vaswani paper doc with exact keyword matches). Lower α degrades results by over-weighting semantic similarity.
- **Semantic query** (`how do neural networks understand context`): Lower α (0.3, more SBERT weight) surfaces the most conceptually relevant document. With α=0.7 BM25 biases towards documents that happen to share surface-level tokens.
- **Default α=0.5 is a reasonable starting point**, but domain-specific tuning on a validation set can yield meaningful gains.

---

## Bonus 2 — Chunk Size Study

We split a long technical paragraph into chunks of 50, 100, and 200 words and
observe how chunk size affects which content is retrieved for the same query.

In [33]:
# Long source document (>500 words)
long_doc = """
The Transformer architecture, introduced by Vaswani et al. in 2017, represented
a fundamental departure from recurrent and convolutional approaches to sequence modelling.
Rather than processing tokens sequentially, the Transformer uses self-attention to relate
all positions in a sequence simultaneously, enabling substantially more parallelism during
training. At the core of the architecture is the scaled dot-product attention function,
which computes a weighted sum of value vectors, with weights determined by the compatibility
between a query vector and a set of key vectors. Multi-head attention extends this by
projecting queries, keys, and values into multiple subspaces and running attention in
parallel, allowing the model to jointly attend to information from different representational
subspaces at different positions. Positional encodings, typically sinusoidal or learned,
are added to the input embeddings to inject information about the order of tokens, since
the self-attention mechanism is inherently permutation-equivariant. The encoder stack
processes an input sequence into a sequence of contextual representations, while the
decoder uses cross-attention to attend to the encoder output and auto-regressively
generates the output sequence one token at a time. Layer normalisation and residual
connections are applied after every sub-layer, stabilising training in deep networks.
The feed-forward sub-layer, applied independently to each position, introduces non-linearity
and increases representational capacity. Variants such as BERT remove the decoder and
pre-train bidirectionally using masked language modelling, while GPT models use only the
decoder in a causal (left-to-right) setting for language generation. Modern architectures
like T5 unify tasks under a text-to-text framework, and models such as PaLM scale
transformers to hundreds of billions of parameters using techniques like mixture-of-experts
routing and pipeline parallelism across thousands of accelerators.
""".strip()


def chunk_text(text: str, chunk_size: int) -> List[str]:
    """Split text into non-overlapping word-count chunks."""
    words  = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i + chunk_size]))
    return chunks


test_query = "how does positional encoding work in transformers?"

print(f"Long doc: {len(long_doc.split())} words\n")

for chunk_size in [50, 100, 200]:
    chunks = chunk_text(long_doc, chunk_size)
    temp_retriever = HybridRetriever(chunks, k=60)
    results = temp_retriever.retrieve(test_query, top_k=1)
    top = results[0]
    print(f"\nChunk size = {chunk_size} words → {len(chunks)} chunks")
    print(f"  Top doc  : {top['text'][:100]}...")
    print(f"  RRF score: {top['rrf_score']:.5f}")

Long doc: 269 words

✅ BM25 index built.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ SBERT index built. Embedding shape: (6, 384)

Chunk size = 50 words → 6 chunks
  Top doc  : the model to jointly attend to information from different representational subspaces at different po...
  RRF score: 0.03279
✅ BM25 index built.


...

✅ SBERT index built. Embedding shape: (3, 384)

Chunk size = 100 words → 3 chunks
  Top doc  : the model to jointly attend to information from different representational subspaces at different po...
  RRF score: 0.03279
✅ BM25 index built.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ SBERT index built. Embedding shape: (2, 384)

Chunk size = 200 words → 2 chunks
  Top doc  : The Transformer architecture, introduced by Vaswani et al. in 2017, represented a fundamental depart...
  RRF score: 0.03252


### Chunk Size Observations

- **Small chunks (50 words):** Higher precision — the retrieved chunk is tightly focused on positional encoding. However, context is stripped away; answers may be incomplete.
- **Medium chunks (100 words):** Good balance — enough context for a complete answer while still targeted enough to avoid retrieval noise.
- **Large chunks (200 words):** The positional encoding sentence is buried in a chunk that also discusses multi-head attention and layer norm, diluting the relevance signal. Dense retrieval scores drop because the embedding averages over many topics.

**Takeaway:** Chunk size is a crucial hyperparameter. For technical Q&A, ~100-word chunks typically balance specificity and context.

---
## Summary

| Component | Implementation | Key Insight |
|---|---|---|
| **BM25** | `rank_bm25.BM25Okapi` on lowercased tokens | Excellent for exact keyword / proper noun matches |
| **SBERT** | `all-MiniLM-L6-v2` cosine similarity | Captures semantic meaning; misses jargon |
| **RRF** | $\frac{1}{k+r_{BM25}} + \frac{1}{k+r_{SBERT}}$ | Robustly fuses two ranked lists without tuning score scales |
| **HyDE** | Gemini 1.5 Flash, `temperature=0.0` | Bridges vocabulary gap between vague questions and precise docs |
| **Cross-Encoder** | `ms-marco-MiniLM-L-6-v2` | Fine-grained re-ranking; more accurate than bi-encoder similarity |
| **Generation** | Groq `llama3-8b-8192` | Fast inference; grounded by top-3 context docs |